# 03e · Train ResNet-18 with a saliency penalty ("right for the right reasons")
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mundmina/dl_project/blob/main/notebooks/03e_train_rrr.ipynb)

**Runtime → T4 GPU → Run all.** 2 конфигурации × 3 seed, ~2.5–3.5 ч (можно в несколько заходов).

Другой подход: не менять данные, а менять функцию потерь (Ross et al., 2017). Модель видит только оригинальные фото,
как A, но к cross-entropy добавляется штраф: доля градиента по входу (saliency), попавшая на фон по маске.
Штраф лежит в [0, 1], поэтому λ сравнима с loss: `resnet_rrr1` (λ = 1) и `resnet_rrr10` (λ = 10).
Обучение в fp32 (двойной backward), поэтому медленнее остальных ResNet-18.
В истории обучения есть колонка `train_bg_share` — она должна падать.

Уже обученные запуски пропускаются, поэтому повторный Run all безопасен.
Если Colab отключился — просто запусти ноутбук снова, обучение продолжится с последней эпохи.
Checkpoints: `MyDrive/bgbias/runs/<name>_seed<k>/`. В конце истории обучения пушатся в GitHub.
После всех 03* ноутбуков — `04_evaluate`, затем `05_analysis`.

In [ ]:
CONFIGS = ['resnet_rrr1', 'resnet_rrr10']
SEEDS = [0, 1, 2]

In [ ]:
# Shared setup: Drive, repo, dependencies. Same cell in every notebook.
# Colab: nothing to set. Local machine (own GPU): see LOCAL.md — open this notebook from the repo's notebooks/
# folder and set BGBIAS_DRIVE, BGBIAS_DATA and GITHUB_TOKEN before starting Jupyter.
REPO_URL = "https://github.com/mundmina/dl_project.git"
BRANCH = "main"

import os, subprocess, sys
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_DIR = "/content/dl_project"
    drive.mount("/content/drive")
    if not os.path.exists(REPO_DIR):
        subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)
    else:  # always run the latest code from GitHub
        subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "checkout", "-B", BRANCH, f"origin/{BRANCH}"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/requirements.txt"], check=True)
else:
    REPO_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))   # notebook opened from <repo>/notebooks
    assert os.path.exists(os.path.join(REPO_DIR, "src", "bgbias")), "open the notebook from the repo's notebooks/ folder"
    missing = [v for v in ("BGBIAS_DRIVE", "BGBIAS_DATA") if not os.environ.get(v)]
    assert not missing, f"set {missing} before starting Jupyter (see LOCAL.md)"
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", BRANCH], check=True)
commit = subprocess.run(["git", "-C", REPO_DIR, "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout
print("code version:", commit)
if f"{REPO_DIR}/src" not in sys.path:
    sys.path.insert(0, f"{REPO_DIR}/src")
# DataLoader workers on Windows start fresh interpreters: they find bgbias through PYTHONPATH, not sys.path
os.environ["PYTHONPATH"] = os.pathsep.join(p for p in (f"{REPO_DIR}/src", os.environ.get("PYTHONPATH")) if p)
try:  # ask for secret access now, not at the end of a long run (the prompt times out unattended)
    if not os.environ.get("GITHUB_TOKEN"):
        from google.colab import userdata
        userdata.get("GITHUB_TOKEN")
    print("GITHUB_TOKEN: ok")
except Exception as e:  # noqa: BLE001
    print(f"GITHUB_TOKEN not available ({type(e).__name__}): results will not be pushed to GitHub")

In [ ]:
import torch
assert torch.cuda.is_available(), "Runtime -> Change runtime type -> T4 GPU"

In [ ]:
from pathlib import Path
from bgbias.project import load_project_data
from bgbias.utils import DRIVE_ROOT

data = load_project_data(REPO_DIR)   # splits / manifests from the repo or from MyDrive/bgbias/week1
RUNS = DRIVE_ROOT / "runs"           # checkpoints and per-run outputs live on Drive
RUNS.mkdir(parents=True, exist_ok=True)
{name: len(df) for name, df in data.splits.items()}

In [ ]:
import pandas as pd
from bgbias.train import load_config, train

configs = {name: load_config(f"{REPO_DIR}/configs/{name}.yaml") for name in CONFIGS}
pd.DataFrame(configs)   # paired configs (X_A / X_B) must differ only in name and p_swap

In [ ]:
# seed-major order: if Colab stops early, every config already has seed 0
histories = {}
for seed in SEEDS:
    for name, cfg in configs.items():
        run_dir = RUNS / f"{name}_seed{seed}"
        print(f"=== {run_dir.name} ===")
        histories[(name, seed)] = pd.DataFrame(train(cfg, seed, data, run_dir))

In [ ]:
rows = []
for (name, seed), h in histories.items():
    best = h.loc[h.val_acc_original.idxmax()]
    rows.append({"model": name, "seed": seed, "best_epoch": int(best.epoch),
                 "val_original": best.val_acc_original, "val_swapped": best.val_acc_swapped,
                 "gap": best.val_acc_original - best.val_acc_swapped})
best = pd.DataFrame(rows)
print(best.groupby("model")[["val_original", "val_swapped", "gap"]].agg(["mean", "std"]).round(4))
best.round(4)

In [ ]:
from bgbias.github_sync import sync
sync(REPO_DIR, "Training histories: " + ", ".join(CONFIGS))